# 01_data_download_binance.ipynb

This notebook downloads historical market data for BTCUSDT from the Binance exchange.

### Objectives:
1. Load configuration parameters.
2. Download OHLCV data for multiple timeframes (1m, 5m, 15m, 1h, 4h, 1d) incrementally.
3. Download futures metrics (funding rates, open interest) if available.
4. Store the raw datasets as Parquet files in Google Drive.

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import sys
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Load Configurations

In [ ]:
from utils.config import load_global_config
from utils.time_utils import timestamp_to_ms

config = load_global_config(PROJECT_ROOT)
symbol = config.get("data", "symbol", "BTCUSDT")
market_type = config.get("data", "market_type", "swap")
start_date = config.get("data", "start_date", "2020-01-01")
debug_mode = config.get("data", "debug_mode", True)
force_rebuild = config.get("data", "force_rebuild", False)

print(f"Symbol: {symbol}")
print(f"Market Type: {market_type}")
print(f"Start Date: {start_date}")
print(f"Debug Mode: {debug_mode}")

## 3. Download OHLCV Data

We download the execution timeframe (5m), the secondary timeframe (15m), and the context timeframes (1h, 4h, 1d). If `debug_mode` is True, we only download a small window (e.g. last 30 days) to avoid long runtimes during development.

In [ ]:
from data.binance_downloader import BinanceDownloader
from data.parquet_store import ParquetStore

downloader = BinanceDownloader(market_type=market_type)
store = ParquetStore(PROJECT_ROOT)

# Determine start timestamp
if debug_mode:
    # In debug mode, start 30 days ago
    start_ts = int((pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=30)).value // 10**6)
else:
    start_ts = timestamp_to_ms(start_date)

timeframes = ["5m", "15m", "1h", "4h", "1d"]

for tf in timeframes:
    # Check if file exists and we are not forcing rebuild
    existing_df = store.load_raw_klines(symbol, tf)
    if existing_df is not None and not existing_df.empty and not force_rebuild:
        # Incremental download: start from the last timestamp in the database
        last_ts = int(existing_df["timestamp"].max())
        print(f"Found existing data for {tf}. Last candle: {pd.to_datetime(last_ts, unit='ms')}. Downloading incrementally...")
        df_new = downloader.download_ohlcv(symbol, tf, start_time_ms=last_ts + 1)
        if not df_new.empty:
            df_combined = pd.concat([existing_df, df_new], ignore_index=True).drop_duplicates(subset=["timestamp"])
            store.save_raw_klines(df_combined, symbol, tf)
        else:
            print(f"No new candles found for {tf}.")
    else:
        # Fresh download
        df = downloader.download_ohlcv(symbol, tf, start_time_ms=start_ts)
        if not df.empty:
            store.save_raw_klines(df, symbol, tf)
        else:
            print(f"Failed to download data for {tf}.")

## 4. Download Futures Metrics (Funding Rate & Open Interest)

In [ ]:
from data.futures_metrics import FuturesMetricsDownloader
from utils.io_utils import save_parquet, load_parquet

metrics_downloader = FuturesMetricsDownloader()

if market_type == "swap":
    # Download Funding Rates
    funding_path = os.path.join(PROJECT_ROOT, "data", "raw", "futures_metrics", f"{symbol}_funding_rates.parquet")
    existing_funding = load_parquet(funding_path)
    
    funding_start = start_ts
    if existing_funding is not None and not existing_funding.empty and not force_rebuild:
        funding_start = int(existing_funding["timestamp"].max()) + 1
        
    df_funding = metrics_downloader.download_funding_rates(symbol, funding_start)
    if not df_funding.empty:
        if existing_funding is not None and not existing_funding.empty:
            df_funding = pd.concat([existing_funding, df_funding], ignore_index=True).drop_duplicates(subset=["timestamp"])
        save_parquet(df_funding, funding_path)
        print(f"Saved funding rates. Total rows: {len(df_funding)}")
        
    # Download Open Interest (on 5m timeframe)
    oi_path = os.path.join(PROJECT_ROOT, "data", "raw", "futures_metrics", f"{symbol}_open_interest.parquet")
    existing_oi = load_parquet(oi_path)
    
    oi_start = start_ts
    if existing_oi is not None and not existing_oi.empty and not force_rebuild:
        oi_start = int(existing_oi["timestamp"].max()) + 1
        
    df_oi = metrics_downloader.download_open_interest(symbol, "5m", oi_start)
    if not df_oi.empty:
        if existing_oi is not None and not existing_oi.empty:
            df_oi = pd.concat([existing_oi, df_oi], ignore_index=True).drop_duplicates(subset=["timestamp"])
        save_parquet(df_oi, oi_path)
        print(f"Saved open interest history. Total rows: {len(df_oi)}")
else:
    print("Spot market selected; skipping futures metrics download.")

## Summary of Downloaded Artifacts

The following raw datasets have been successfully saved to Google Drive:
1. `data/raw/klines/BTCUSDT_{timeframe}_raw.parquet` (for 5m, 15m, 1h, 4h, 1d)
2. `data/raw/futures_metrics/BTCUSDT_funding_rates.parquet` (if Futures)
3. `data/raw/futures_metrics/BTCUSDT_open_interest.parquet` (if Futures)

**Next Step**: Run [02_data_cleaning_and_resampling.ipynb](file:///content/drive/MyDrive/btcusdt_quant_research/notebooks/02_data_cleaning_and_resampling.ipynb) to clean and resample the datasets.